# Incident assistant

This notebook runs the local LangGraph path using the saved anomaly and log models and the FAISS index. It makes no external API request.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "requirements-assistant.txt").exists():
    if root.parent == root:
        raise FileNotFoundError("Run this notebook within the repository")
    root = root.parent
sys.path.insert(0, str(root))
from banking_observability.incident_assistant import run_assistant, create_workflow
root

## Graph nodes

`gather` loads the evidence packet, `reduce` selects source-linked items, and `finalize` assembles the local response. Selecting a model adds a generation step before finalization.

In [ ]:
graph = create_workflow()
list(graph.nodes)

## Replaying a recorded minute

The monitoring review flag indicates an anomaly check, not an established incident. Metric values and log messages retain their source rows.

In [ ]:
result = run_assistant(
    question="network timeout latency", as_of="2025-10-25 18:01:00",
    data_dir=root / "data/raw", knowledge_dir=root / "models/knowledge",
    log_model_dir=root / "models/log_clustering",
    anomaly_model_dir=root / "models/detection/monitoring_anomaly",
)
result

## Reading the output

Compare the values to the original source minute and inspect the selected logs and runbook references. The assistant does not establish a cause or a resolution. External generation is available through the CLI only after explicit approval to transmit bounded source material.

In [ ]:
result["monitoring_evidence"], result["recent_log_evidence"], result["runbook_evidence"]